# Literary Psychology — Predictive Modelling & Embedding Analysis
**K. Sh. Karimov, 2026**

Companion notebook to `01_eda_and_stats.ipynb`.  
Requires `authors_annotated.json` in the same directory.

> **Recommended runtime:** Google Colab with T4 GPU (~20 min total)  
> Local CPU is possible but embedding encoding will take ~60 min.

---
## Pipeline
```
1. Setup & data loading
2. Tag-based logistic regression      (AUC = 0.941 ± 0.015)  ← from 01_eda
3. Biography embeddings               (nomic-embed-text-v1.5, 8192 tokens)
4. Model comparison                   (embeddings vs tags vs TF-IDF)
5. UMAP visualisation                 (semantic space of biographies)
6. SHAP: words driving high score     (TF-IDF interpretability)
7. Mediation analysis                 (childhood_trauma → mediator → score)
8. Summary & export
```

## Key results (run to reproduce)
| Model | CV ROC-AUC |
|-------|-----------|
| Binary tags (32) | **0.947 ± 0.017** |
| TF-IDF bag-of-words | 0.846 ± 0.038 |
| Embeddings (nomic-v1.5) | 0.787 ± 0.030 |

Mediation: childhood_trauma → depression → score (suppression effect, 109.8%)


---
## 1. Setup

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
# On Colab: Runtime → Change runtime type → T4 GPU before running
!pip install -q sentence-transformers einops umap-learn shap pingouin

import json, warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

import torch
warnings.filterwarnings('ignore')

FIGS = Path('figures_emb')
FIGS.mkdir(exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
print('Setup complete.')


---
## 2. Data loading

In [ ]:
# ── Load dataset ──────────────────────────────────────────────────────────
# On Colab: upload authors_annotated.json when prompted
import os

if not os.path.exists('authors_annotated.json'):
    from google.colab import files
    print('Upload authors_annotated.json:')
    files.upload()

with open('authors_annotated.json') as f:
    df = pd.DataFrame(json.load(f))

ERA_MAP = {'19': 'XIX', '20': 'XX', '21': 'XXI'}
df['era'] = df['era'].astype(str).map(ERA_MAP).fillna('Unknown')

df['score'] = pd.to_numeric(df['standardness_score'], errors='coerce')
df['conf']  = pd.to_numeric(df['confidence'], errors='coerce')

gender_map = {'male': 'M', 'female': 'F', 'unknown': 'Unknown'}
df['gender'] = df['gender'].map(gender_map).fillna('Unknown')

TAGS = [
    'tag_depression','tag_bipolar','tag_schizophrenia','tag_anxiety',
    'tag_ptsd','tag_substance_abuse','tag_suicide','tag_suicide_attempt',
    'tag_institutionalized','tag_childhood_trauma','tag_war_experience',
    'tag_poverty_extreme','tag_chronic_illness','tag_disability',
    'tag_occultism','tag_spiritualism','tag_religious_mania',
    'tag_cult_involvement','tag_theosophy_mysticism',
    'tag_non_traditional_relationship','tag_homosexuality_taboo_era',
    'tag_obsessive_attachment','tag_celibacy_pathological',
    'tag_incest_adjacent','tag_alter_ego_documented',
    'tag_depersonalization','tag_voluntary_isolation',
    'tag_pathological_gambling','tag_legal_troubles',
    'tag_imprisonment','tag_exile','tag_extremist_views',
    'tag_violence_documented','tag_self_destructive_pattern',
    'tag_eating_disorder','tag_paranoia','tag_messiah_complex',
    'tag_nihilism_explicit'
]

bool_map = {True: True, False: False, 'true': True, 'false': False,
            1: True, 0: False, 'True': True, 'False': False,
            'null': False, None: False}
for t in TAGS:
    if t in df.columns:
        df[t] = df[t].map(bool_map).fillna(False).astype(bool)

df = df[df['score'].notna() & df['bio_text'].notna()].copy()
df = df[df['bio_text'].str.len() > 200].copy()
df = df.reset_index(drop=True)

print(f'Authors : {len(df)}')
print(f'Bio median length : {df["bio_text"].str.len().median():,.0f} chars')
print(df['era'].value_counts())


---
## 3. Tag-based logistic regression (baseline)
Recap from `01_eda_and_stats.ipynb` §14.  
CV AUC = **0.941 ± 0.015** — strong baseline that embedding models must beat.


In [ ]:
# ── 3.1  Tag-based logistic regression ────────────────────────────────────
from sklearn.model_selection import StratifiedKFold, cross_val_score

y_bin = (df['score'] >= 5).astype(int)

cluster_tags = [t for t in TAGS if t in df.columns and df[t].sum() >= 10]
X_tags = df[cluster_tags].astype(int).values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_tags = Pipeline([
    ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42))
])

auc_tags = cross_val_score(pipe_tags, X_tags, y_bin, cv=cv, scoring='roc_auc')
acc_tags = cross_val_score(pipe_tags, X_tags, y_bin, cv=cv, scoring='accuracy')

print('=== Tag-based model (baseline) ===')
print(f'CV ROC-AUC : {auc_tags.mean():.3f} ± {auc_tags.std():.3f}')
print(f'CV Accuracy: {acc_tags.mean():.3f} ± {acc_tags.std():.3f}')
print(f'Features   : {X_tags.shape[1]} binary tags')


---
## 4. Biography embeddings
**Model:** `nomic-ai/nomic-embed-text-v1.5`
- Context: **8192 tokens** (~30 000 chars) vs 384 tokens for standard models
- Dimension: 768
- Biographies > 30k chars (29%) handled by **chunk-and-mean pooling**


In [ ]:
# ── 4.1  Load model ────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    'nomic-ai/nomic-embed-text-v1.5',
    trust_remote_code=True,
    device=device
)
model.max_seq_length = 8192

print(f'Model   : nomic-embed-text-v1.5')
print(f'Context : {model.max_seq_length} tokens (~30 000 chars)')
print(f'Emb dim : {model.get_sentence_embedding_dimension()}')


In [ ]:
# ── 4.2  Bio length audit ─────────────────────────────────────────────────
lengths = df['bio_text'].str.len()
n_single = (lengths <= 30000).sum()
n_multi  = (lengths > 30000).sum()

print('Bio length stats (chars):')
print(f'  Median : {lengths.median():,.0f}')
print(f'  p75    : {lengths.quantile(0.75):,.0f}')
print(f'  p90    : {lengths.quantile(0.90):,.0f}')
print(f'  Max    : {lengths.max():,.0f}')
print()
print(f'Single-chunk (<=30k) : {n_single} ({n_single/len(df)*100:.1f}%)')
print(f'Multi-chunk  (>30k)  : {n_multi}  ({n_multi/len(df)*100:.1f}%)')
print('Coverage: 100% via chunk-and-mean')


In [ ]:
# ── 4.3  Encode with chunk-and-mean ───────────────────────────────────────
# Bio <= 30k: single pass
# Bio >  30k: split into overlapping chunks, encode each, mean-pool
CHUNK_SIZE = 30000
OVERLAP    = 500
BATCH_SIZE = 4

def encode_with_chunking(texts, model, chunk_size=CHUNK_SIZE,
                          overlap=OVERLAP, batch_size=BATCH_SIZE):
    all_vecs = []
    step = chunk_size - overlap

    for idx, text in enumerate(texts):
        if idx % 50 == 0:
            print(f'  [{idx}/{len(texts)}]', end='\r')

        chunks = [text] if len(text) <= chunk_size else [
            text[s:s+chunk_size]
            for s in range(0, len(text), step)
            if len(text[s:s+chunk_size]) > 100
        ]

        chunk_vecs = []
        for i in range(0, len(chunks), batch_size):
            vecs = model.encode(
                chunks[i:i+batch_size],
                batch_size=batch_size,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            chunk_vecs.append(vecs)

        chunk_vecs = np.vstack(chunk_vecs)
        mean_vec = chunk_vecs.mean(axis=0)
        mean_vec = mean_vec / (np.linalg.norm(mean_vec) + 1e-9)
        all_vecs.append(mean_vec)

    print(f'  [{len(texts)}/{len(texts)}] done.')
    return np.vstack(all_vecs)


print(f'Encoding {len(df)} biographies...')
embeddings = encode_with_chunking(
    df['bio_text'].fillna('').str[:100000].tolist(),
    model
)

print(f'\nEmbeddings shape: {embeddings.shape}')
np.save('embeddings.npy', embeddings)
print('Saved: embeddings.npy')


---
## 5. Model comparison

In [ ]:
# ── 5.1  Prepare embedding features ───────────────────────────────────────
X_emb = embeddings   # 768-dim

print(f'Embedding features : {X_emb.shape[1]}')
print(f'Tag features       : {X_tags.shape[1]}')
print(f'Target balance     : {y_bin.mean()*100:.1f}% high-score (>=5)')


In [ ]:
# ── 5.2  Cross-validated AUC: three models ────────────────────────────────
# Model A: embeddings + logistic regression
pipe_emb = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, C=1.0, random_state=42))
])

# Model C: TF-IDF on raw bio_text (bag-of-words, no pretrained model)
tfidf_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2),
                              stop_words='english', sublinear_tf=True)),
    ('scaler', StandardScaler(with_mean=False)),
    ('clf',    LogisticRegression(max_iter=1000, C=1.0, random_state=42))
])

bio_texts = df['bio_text'].fillna('').str[:30000]

print('Running 5-fold CV...')
auc_emb   = cross_val_score(pipe_emb,   X_emb,     y_bin, cv=cv, scoring='roc_auc')
auc_tfidf = cross_val_score(tfidf_pipe, bio_texts,  y_bin, cv=cv, scoring='roc_auc')

print()
print('=== Model comparison (5-fold CV ROC-AUC) ===')
print(f'A. Binary tags (32)          : {auc_tags.mean():.3f} ± {auc_tags.std():.3f}  ← best')
print(f'B. Embeddings (nomic-v1.5)   : {auc_emb.mean():.3f} ± {auc_emb.std():.3f}')
print(f'C. TF-IDF bag-of-words       : {auc_tfidf.mean():.3f} ± {auc_tfidf.std():.3f}')
print()
print('Interpretation:')
print('  Tags beat raw text → structured annotation captures signal')
print('  that surface-level language cannot.')
print('  Embeddings (0.787) >> random (0.5) → biographical text does')
print('  encode psychological signal, just less precisely than expert tags.')


In [ ]:
# ── 5.3  ROC curves ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

models_to_plot = [
    ('Binary tags (32)',       pipe_tags,   X_tags,    '#1D9E75'),
    ('Embeddings (nomic-v1.5)', pipe_emb,   X_emb,     '#378ADD'),
    ('TF-IDF BoW',             tfidf_pipe,  bio_texts, '#D85A30'),
]

for name, pipe, X, col in models_to_plot:
    pipe.fit(X, y_bin)
    y_prob = pipe.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y_bin, y_prob)
    auc = roc_auc_score(y_bin, y_prob)
    ax.plot(fpr, tpr, lw=2, color=col, label=f'{name}  (full AUC={auc:.3f})')

ax.plot([0,1],[0,1], 'k--', lw=0.8, label='Random')
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC curves — three models\n(full-data; report CV AUC in paper)',
             fontweight='500')
ax.legend(fontsize=9)
ax.text(0.5, 0.05,
        '⚠  Full-data ROC is optimistic. CV AUC in legend above.',
        transform=ax.transAxes, fontsize=8, color='gray', style='italic')
ax.set_facecolor('white'); fig.patch.set_facecolor('white')
plt.tight_layout()
plt.savefig(FIGS/'fig_roc_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_roc_comparison.png')


---
## 6. UMAP visualisation
Project 768-dim embeddings to 2D. If high-score authors cluster together,
psychological profile is geometrically encoded in biography space.

In [ ]:
# ── 6.1  Fit UMAP ─────────────────────────────────────────────────────────
import umap

reducer = umap.UMAP(
    n_components=2, n_neighbors=15,
    min_dist=0.1, metric='cosine',
    random_state=42
)
coords_2d = reducer.fit_transform(embeddings)
df['umap1'] = coords_2d[:, 0]
df['umap2'] = coords_2d[:, 1]
print('UMAP done.')


In [ ]:
# ── 6.2  UMAP plots ───────────────────────────────────────────────────────
ERA_COLORS = {'XIX': '#3B6D11', 'XX': '#185FA5', 'XXI': '#993C1D'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: colored by score
ax = axes[0]
sc = ax.scatter(df['umap1'], df['umap2'], c=df['score'],
                cmap='RdYlGn_r', s=12, alpha=0.7)
plt.colorbar(sc, ax=ax, label='Score')
ax.set_title('UMAP — colored by score', fontweight='500')
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')

# Panel 2: colored by era
ax = axes[1]
for era, col in ERA_COLORS.items():
    m = df['era'] == era
    ax.scatter(df.loc[m,'umap1'], df.loc[m,'umap2'],
               c=col, s=12, alpha=0.65, label=f'{era} (n={m.sum()})')
ax.set_title('UMAP — colored by era', fontweight='500')
ax.set_xlabel('UMAP 1'); ax.legend(fontsize=8)

# Panel 3: top-20 highest score authors labelled
ax = axes[2]
ax.scatter(df['umap1'], df['umap2'], c=df['score'],
           cmap='RdYlGn_r', s=10, alpha=0.5)
for _, row in df.nlargest(20, 'score').iterrows():
    ax.annotate(
        row['author_name'].split()[-1],
        (row['umap1'], row['umap2']),
        fontsize=6.5, color='#1a1a2e',
        xytext=(3, 3), textcoords='offset points'
    )
ax.set_title('UMAP — top-20 highest score authors', fontweight='500')
ax.set_xlabel('UMAP 1')

for a in axes: a.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.savefig(FIGS/'fig_umap.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_umap.png')


---
## 7. SHAP — words driving high/low score
TF-IDF logistic regression + LinearExplainer.
Shows which words and bigrams in biographies are most associated
with high psychological non-standardness.

**Known limitation:** TF-IDF partially captures era (XIX/XX authors have
richer biographies with different vocabulary than XXI authors).
Words like *french*, *paris*, *poet* may reflect era as much as psychology.


In [ ]:
# ── 7.1  Fit TF-IDF + LogReg ──────────────────────────────────────────────
import shap

tfidf_vec = TfidfVectorizer(
    max_features=3000, ngram_range=(1,2),
    stop_words='english', sublinear_tf=True
)
X_tfidf = tfidf_vec.fit_transform(df['bio_text'].fillna('').str[:30000])

lr_tfidf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_tfidf.fit(X_tfidf, y_bin)

print(f'Vocabulary size: {len(tfidf_vec.vocabulary_)}')


In [ ]:
# ── 7.2  SHAP values ──────────────────────────────────────────────────────
explainer = shap.LinearExplainer(
    lr_tfidf, X_tfidf, feature_perturbation='interventional'
)
shap_values = explainer.shap_values(X_tfidf)

feature_names = tfidf_vec.get_feature_names_out()
mean_shap = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    'feature'   : feature_names,
    'mean_shap' : mean_shap,
    'coef'      : lr_tfidf.coef_[0]
}).sort_values('mean_shap', ascending=False)

print('Top-20 words/bigrams by SHAP importance:')
print(shap_df.head(20)[['feature','mean_shap','coef']].round(4).to_string(index=False))
print()
print('NOTE: year tokens (2010, 2019) and "official website" are era markers')
print('for XXI authors — they capture documentation style, not psychology directly.')


In [ ]:
# ── 7.3  Plot top-30 SHAP features ────────────────────────────────────────
top30 = shap_df.head(30).sort_values('coef')

fig, ax = plt.subplots(figsize=(10, 9))
colors = ['#1D9E75' if v > 0 else '#E24B4A' for v in top30['coef']]
ax.barh(top30['feature'], top30['coef'], color=colors, height=0.7, alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Logistic regression coefficient (TF-IDF model)')
ax.set_title(
    'Top-30 words/bigrams associated with high psychological score\n'
    '(green = high score, red = low score)',
    fontweight='500'
)
ax.set_facecolor('white'); fig.patch.set_facecolor('white')
plt.tight_layout()
plt.savefig(FIGS/'fig_shap_words.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_shap_words.png')


---
## 8. Mediation analysis
**Question:** Does childhood trauma affect score DIRECTLY,
or is the effect mediated through depression and self-destructive patterns?

```
childhood_trauma ──→ depression          ──→ score   (indirect)
childhood_trauma ──────────────────────→ score       (direct)
```

**Suppression effect** (proportion > 100%) means the mediator carries
more of the effect than the total — the direct path partially suppresses
the mediated path.


In [ ]:
# ── 8.1  Mediation analysis ───────────────────────────────────────────────
import pingouin as pg

med_df = df[['tag_childhood_trauma','tag_depression',
             'tag_self_destructive_pattern','score']].copy()

for col in ['tag_childhood_trauma','tag_depression','tag_self_destructive_pattern']:
    med_df[col] = med_df[col].astype(int)
med_df = med_df.dropna()

print('=== Mediation 1: childhood_trauma → depression → score ===')
med1 = pg.mediation_analysis(
    data=med_df, x='tag_childhood_trauma',
    m='tag_depression', y='score',
    n_boot=1000, seed=42
)
print(med1.to_string(index=False))

print()
print('=== Mediation 2: childhood_trauma → self_destructive_pattern → score ===')
med2 = pg.mediation_analysis(
    data=med_df, x='tag_childhood_trauma',
    m='tag_self_destructive_pattern', y='score',
    n_boot=1000, seed=42
)
print(med2.to_string(index=False))


In [ ]:
# ── 8.2  Mediation path visualisation ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, med, mediator in zip(
    axes,
    [med1, med2],
    ['depression', 'self_destructive']
):
    total  = med.loc[med['path']=='Total',    'coef'].values[0]
    direct = med.loc[med['path']=='Direct',   'coef'].values[0]
    indir  = med.loc[med['path']=='Indirect', 'coef'].values[0]
    prop   = indir / total * 100 if total != 0 else 0

    ax.bar(
        ['Total effect', 'Direct effect', 'Indirect\n(mediated)'],
        [total, direct, indir],
        color=['#378ADD','#1D9E75','#D85A30'],
        alpha=0.85, width=0.5
    )
    ax.axhline(0, color='black', lw=0.8)
    ax.set_ylabel('Coefficient (OLS)')
    ax.set_title(
        f'childhood_trauma → {mediator} → score\n'
        f'Mediated proportion: {prop:.1f}%',
        fontweight='500'
    )
    ax.set_facecolor('white')

fig.patch.set_facecolor('white')
plt.tight_layout()
plt.savefig(FIGS/'fig_mediation.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: fig_mediation.png')


---
## 9. Summary & export

In [ ]:
# ── 9.1  Results summary ──────────────────────────────────────────────────
print('=' * 60)
print('MODELLING SUMMARY')
print('=' * 60)
print()
print('--- Model comparison (5-fold CV ROC-AUC) ---')
print(f'  Binary tags (32)         : {auc_tags.mean():.3f} ± {auc_tags.std():.3f}  ← best')
print(f'  TF-IDF bag-of-words      : {auc_tfidf.mean():.3f} ± {auc_tfidf.std():.3f}')
print(f'  Embeddings (nomic-v1.5)  : {auc_emb.mean():.3f} ± {auc_emb.std():.3f}')
print()
print('--- Top-5 words → HIGH score (TF-IDF) ---')
for _, r in shap_df[shap_df['coef']>0].head(5).iterrows():
    print(f'  {r["feature"]:30s}  coef={r["coef"]:+.3f}')
print()
print('--- Top-5 words → LOW score (TF-IDF) ---')
for _, r in shap_df[shap_df['coef']<0].sort_values('coef').head(5).iterrows():
    print(f'  {r["feature"]:30s}  coef={r["coef"]:+.3f}')
print()
print('--- Figures saved ---')
for p in sorted(FIGS.iterdir()):
    print(f'  {p.name}')


In [ ]:
# ── 9.2  Save embeddings for reuse ───────────────────────────────────────
# embeddings.npy already saved in §4.3
# To reload without re-encoding:
#   embeddings = np.load('embeddings.npy')

print('All figures saved to figures_emb/')
print('Embeddings saved to embeddings.npy')
print()
print('On Colab — download with:')
print('  from google.colab import files')
print('  files.download("embeddings.npy")')
